# IPL Data Analysis (Python: NumPy, Pandas, Matplotlib, Seaborn)
**Input files needed in the same folder:** `matches.csv` and `deliveries.csv` (Kaggle "IPL Complete Dataset").
**Outputs:** `IPL_Cleaned.csv` (ball-by-ball), `IPL_Matches_Cleaned.csv` (one row per match), `charts/*.png`

## PART 1 - DATA LOADING

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
os.makedirs("charts", exist_ok=True)

try:
    show = display          # works in Jupyter / Colab
except NameError:
    show = print            # works in plain .py

matches = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

# Rename columns to the names used in the project brief (handles old and new Kaggle versions)
matches = matches.rename(columns={
    "id": "Match_ID", "season": "Season", "date": "Match_Date", "city": "City", "venue": "Venue",
    "team1": "Team1", "team2": "Team2", "toss_winner": "Toss_Winner", "toss_decision": "Toss_Decision",
    "winner": "Winner", "win_by_runs": "Win_By_Runs", "win_by_wickets": "Win_By_Wickets",
    "player_of_match": "Player_of_Match", "result": "Result"})
deliveries = deliveries.rename(columns={
    "match_id": "Match_ID", "inning": "Innings", "batting_team": "Batting_Team",
    "bowling_team": "Bowling_Team", "over": "Over", "ball": "Ball", "batsman": "Batsman",
    "batter": "Batsman", "bowler": "Bowler", "batsman_runs": "Runs_Scored",
    "batter_runs": "Runs_Scored", "extra_runs": "Extra_Runs", "total_runs": "Total_Runs"})

# Newer dataset versions store a single "result_margin" instead of two columns
if "Win_By_Runs" not in matches.columns and "result_margin" in matches.columns:
    matches["Win_By_Runs"] = np.where(matches["Result"] == "runs", matches["result_margin"], 0)
    matches["Win_By_Wickets"] = np.where(matches["Result"] == "wickets", matches["result_margin"], 0)

print("Matches first 5:");  show(matches.head())
print("Matches last 5:");   show(matches.tail())
print("Deliveries first 5:"); show(deliveries.head())
print("Deliveries last 5:");  show(deliveries.tail())
print("Shapes ->", "matches:", matches.shape, "| deliveries:", deliveries.shape)
print("Matches columns:", list(matches.columns))
print("Deliveries columns:", list(deliveries.columns))
print("Numerical (matches):", matches.select_dtypes(include=np.number).columns.tolist())
print("Categorical (matches):", matches.select_dtypes(exclude=np.number).columns.tolist())
print("Numerical (deliveries):", deliveries.select_dtypes(include=np.number).columns.tolist())
print("Categorical (deliveries):", deliveries.select_dtypes(exclude=np.number).columns.tolist())

## PART 2 - DATA EXPLORATION & CLEANING

In [ ]:
print(matches.dtypes); print(deliveries.dtypes)
show(matches.describe()); show(deliveries.describe())
print("Missing values (matches):\n", matches.isnull().sum())
print("Missing values (deliveries):\n", deliveries.isnull().sum())
print("Duplicates -> matches:", matches.duplicated().sum(), "| deliveries:", deliveries.duplicated().sum())

matches = matches.drop_duplicates()
deliveries = deliveries.drop_duplicates()

for col in ["Season", "City", "Venue", "Toss_Decision", "Result"]:
    print(f"\nUnique {col}:", sorted(matches[col].dropna().astype(str).unique()))
print("\nTeam names BEFORE fixing:", sorted(set(matches["Team1"]) | set(matches["Team2"])))

In [ ]:
# Fix inconsistent team names (franchise renames / spelling variants)
team_fix = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Deccan Chargers": "Sunrisers Hyderabad",
    "Royal Challengers Bengaluru": "Royal Challengers Bangalore",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
    "Pune Warriors": "Pune Warriors India",
}
for c in ["Team1", "Team2", "Toss_Winner", "Winner"]:
    matches[c] = matches[c].replace(team_fix)
for c in ["Batting_Team", "Bowling_Team"]:
    deliveries[c] = deliveries[c].replace(team_fix)
print("Team names AFTER fixing:", sorted(set(matches["Team1"]) | set(matches["Team2"])))

# Missing values
matches["Winner"] = matches["Winner"].fillna("No Result")
matches["Player_of_Match"] = matches["Player_of_Match"].fillna("Not Awarded")
matches["City"] = matches["City"].fillna("Unknown")
matches[["Win_By_Runs", "Win_By_Wickets"]] = matches[["Win_By_Runs", "Win_By_Wickets"]].fillna(0)
for c in ["Extra_Runs", "Runs_Scored", "Total_Runs"]:
    deliveries[c] = deliveries[c].fillna(0)

# Dates -> datetime ; Season -> calendar year (fixes labels like "2007/08")
matches["Match_Date"] = pd.to_datetime(matches["Match_Date"], errors="coerce")
matches["Season"] = matches["Match_Date"].dt.year

# Ignore super overs (innings 3 and 4) so they do not distort runs
if "Innings" in deliveries.columns:
    deliveries = deliveries[deliveries["Innings"] <= 2]

# Wickets credited to the bowler (exclude run outs, retired etc.)
if "dismissal_kind" in deliveries.columns:
    bowler_kinds = ["bowled", "caught", "lbw", "stumped", "caught and bowled", "hit wicket"]
    deliveries["Wickets"] = deliveries["dismissal_kind"].isin(bowler_kinds).astype(int)
else:
    deliveries["Wickets"] = deliveries["is_wicket"].fillna(0).astype(int)

# Build the single cleaned dataset requested by the brief (ball-by-ball + match info)
match_cols = ["Match_ID", "Season", "Match_Date", "City", "Venue", "Team1", "Team2", "Toss_Winner",
              "Toss_Decision", "Winner", "Win_By_Runs", "Win_By_Wickets", "Player_of_Match", "Result"]
ipl = deliveries.merge(matches[match_cols], on="Match_ID", how="left")
keep = match_cols + ["Innings", "Batting_Team", "Bowling_Team", "Batsman", "Bowler",
                     "Runs_Scored", "Extra_Runs", "Total_Runs", "Wickets", "Over", "Ball"]
ipl = ipl[keep]

print("Cleaned dataset shape:", ipl.shape)
print("Remaining missing values:\n", ipl.isnull().sum())
show(ipl.head())

ipl.to_csv("IPL_Cleaned.csv", index=False)
matches.to_csv("IPL_Matches_Cleaned.csv", index=False)

## PART 3 - DATA ANALYSIS (Pandas & NumPy)

In [ ]:
valid = matches[matches["Winner"] != "No Result"]          # matches with a winner

print("18. Total matches:", matches["Match_ID"].nunique())
matches_per_season = matches.groupby("Season")["Match_ID"].nunique()
print("19. Matches per season:\n", matches_per_season)
wins = valid["Winner"].value_counts()
print("20. Wins by team:\n", wins)
print("21. Most wins:", wins.idxmax(), wins.max())
print("22. Fewest wins:", wins.idxmin(), wins.min())
pom = matches[matches["Player_of_Match"] != "Not Awarded"]["Player_of_Match"].value_counts()
print("23. Top Player of the Match:", pom.idxmax(), pom.max())
venue_counts = matches["Venue"].value_counts()
print("24. Most used venue:", venue_counts.idxmax(), venue_counts.max())
toss_dec = matches["Toss_Decision"].value_counts()
print("25. Toss decisions:\n", toss_dec)
toss_win_match = (valid["Toss_Winner"] == valid["Winner"])
toss_pct = round(toss_win_match.mean() * 100, 2)
print("26. % matches won after winning toss:", toss_pct)
r = matches.loc[matches["Win_By_Runs"].idxmax()]
print("27. Highest win by runs:", r["Win_By_Runs"], "-", r["Winner"], r["Match_Date"].date())
w = matches.loc[matches["Win_By_Wickets"].idxmax()]
print("28. Highest win by wickets:", w["Win_By_Wickets"], "-", w["Winner"], w["Match_Date"].date())

runs_by_bat = deliveries.groupby("Batsman")["Runs_Scored"].sum().sort_values(ascending=False)
wkts_by_bowl = deliveries.groupby("Bowler")["Wickets"].sum().sort_values(ascending=False)
print("29. Highest run scorer:", runs_by_bat.idxmax(), runs_by_bat.max())
print("30. Most wickets:", wkts_by_bowl.idxmax(), wkts_by_bowl.max())
top10_bat = runs_by_bat.head(10); top10_bowl = wkts_by_bowl.head(10)
print("31. Top 10 batsmen:\n", top10_bat)
print("32. Top 10 bowlers:\n", top10_bowl)

season_runs = ipl.groupby("Season")["Total_Runs"].sum()
season_wkts = ipl.groupby("Season")["Wickets"].sum()
print("33. Season-wise runs:\n", season_runs)
print("34. Season-wise wickets:\n", season_wkts)
avg_runs_match = round(ipl["Total_Runs"].sum() / ipl["Match_ID"].nunique(), 2)
print("35. Average runs per match:", avg_runs_match)

innings_df = (deliveries.groupby(["Match_ID", "Innings", "Batting_Team"])
              .agg(Runs=("Total_Runs", "sum"), Wickets=("Wickets", "sum"),
                   Extras=("Extra_Runs", "sum"), Balls=("Ball", "count")).reset_index())
team_avg = innings_df.groupby("Batting_Team")["Runs"].mean().sort_values(ascending=False).round(2)
print("36. Team-wise average runs per innings:\n", team_avg)

season_team = valid.pivot_table(index="Winner", columns="Season", values="Match_ID",
                                aggfunc="count", fill_value=0)
print("37. Season-wise team wins:"); show(season_team)

best_venue = (valid.groupby(["Winner", "Venue"]).size().reset_index(name="Wins")
              .sort_values("Wins", ascending=False).drop_duplicates("Winner")
              .sort_values("Wins", ascending=False))
print("38. Most successful venue per team:"); show(best_venue)

print("39. Toss outcome vs result:")
print(pd.crosstab(valid["Toss_Winner"] == valid["Winner"], valid["Toss_Decision"],
                  rownames=["Toss winner also won match"], margins=True))
bat_first_win = np.where(valid["Win_By_Runs"] > 0, 1, 0).mean() * 100
print(f"Matches won by team batting first: {bat_first_win:.1f}%")

print("40. Trends -> avg runs/match by season:")
print((season_runs / matches_per_season).round(1))

## PART 4 - PYTHON VISUALIZATIONS

In [ ]:
def save(name):
    plt.tight_layout(); plt.savefig(f"charts/{name}.png", bbox_inches="tight"); plt.show()

# 1 Matches by season
plt.figure(figsize=(10, 5)); sns.barplot(x=matches_per_season.index, y=matches_per_season.values, color="steelblue")
plt.title("Matches Played by Season"); plt.xlabel("Season"); plt.ylabel("Matches"); plt.xticks(rotation=45); save("01_matches_by_season")

# 2 Matches won by team
plt.figure(figsize=(10, 6)); sns.barplot(x=wins.values, y=wins.index, palette="viridis")
plt.title("Matches Won by Team"); plt.xlabel("Wins"); plt.ylabel("Team"); save("02_wins_by_team")

# 3 Top 10 run scorers
plt.figure(figsize=(9, 5)); sns.barplot(x=top10_bat.values, y=top10_bat.index, palette="Blues_r")
plt.title("Top 10 Run Scorers"); plt.xlabel("Runs"); plt.ylabel(""); save("03_top10_batsmen")

# 4 Top 10 wicket takers
plt.figure(figsize=(9, 5)); sns.barplot(x=top10_bowl.values, y=top10_bowl.index, palette="Reds_r")
plt.title("Top 10 Wicket Takers"); plt.xlabel("Wickets"); plt.ylabel(""); save("04_top10_bowlers")

# 5 Player of the match
top_pom = pom.head(10)
plt.figure(figsize=(10, 5)); sns.barplot(x=top_pom.index, y=top_pom.values, palette="mako")
plt.title("Player of the Match Awards (Top 10)"); plt.ylabel("Awards"); plt.xticks(rotation=45, ha="right"); save("05_player_of_match")

# 6 Toss decision pie
plt.figure(figsize=(6, 6)); plt.pie(toss_dec.values, labels=toss_dec.index, autopct="%1.1f%%", startangle=90,
                                    colors=["#4c72b0", "#dd8452"])
plt.title("Toss Decision Distribution"); save("06_toss_decision")

# 7 Matches won after winning toss
toss_won = valid[valid["Toss_Winner"] == valid["Winner"]]["Winner"].value_counts()
plt.figure(figsize=(10, 6)); sns.barplot(x=toss_won.values, y=toss_won.index, palette="crest")
plt.title("Matches Won After Winning the Toss"); plt.xlabel("Matches"); save("07_won_after_toss")

# 8 Season-wise runs
plt.figure(figsize=(10, 5)); plt.plot(season_runs.index, season_runs.values, marker="o", color="green")
plt.title("Season-wise Total Runs"); plt.xlabel("Season"); plt.ylabel("Runs"); plt.grid(True); save("08_season_runs")

# 9 Season-wise wickets
plt.figure(figsize=(10, 5)); plt.plot(season_wkts.index, season_wkts.values, marker="s", color="crimson")
plt.title("Season-wise Total Wickets"); plt.xlabel("Season"); plt.ylabel("Wickets"); plt.grid(True); save("09_season_wickets")

# 10 Runs distribution (innings totals)
plt.figure(figsize=(9, 5)); sns.histplot(innings_df["Runs"], bins=25, kde=True, color="purple")
plt.title("Distribution of Runs per Innings"); plt.xlabel("Runs"); save("10_runs_distribution")

# 11 Team performance comparison (win %)
played = pd.concat([matches["Team1"], matches["Team2"]]).value_counts()
win_pct = (wins / played * 100).dropna().sort_values(ascending=False)
plt.figure(figsize=(10, 6)); sns.barplot(x=win_pct.values, y=win_pct.index, palette="coolwarm_r")
plt.title("Team Performance Comparison (Win %)"); plt.xlabel("Win %"); save("11_team_win_pct")

# 12 Venue-wise match count
top_venues = venue_counts.head(10)
plt.figure(figsize=(10, 6)); sns.barplot(x=top_venues.values, y=top_venues.index, palette="rocket")
plt.title("Venue-wise Match Count (Top 10)"); plt.xlabel("Matches"); save("12_venue_matches")

# 13 Runs vs wickets
plt.figure(figsize=(8, 6)); sns.scatterplot(data=innings_df, x="Wickets", y="Runs", alpha=0.5)
plt.title("Runs vs Wickets (per Innings)"); save("13_runs_vs_wickets")

# 14 Correlation heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(innings_df[["Runs", "Wickets", "Extras", "Balls"]].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap"); save("14_correlation_heatmap")

## PART 6 - BUSINESS INSIGHTS (auto-generated from your data)

In [ ]:
season_avg = (season_runs / matches_per_season)
print("1.  Highest wins:", wins.idxmax(), f"({wins.max()})")
print("2.  Season with most matches:", matches_per_season.idxmax(), f"({matches_per_season.max()})")
print("3.  Most Player of the Match awards:", pom.idxmax(), f"({pom.max()})")
print("4.  Highest run scorer:", runs_by_bat.idxmax(), f"({runs_by_bat.max()})")
print("5.  Highest wicket taker:", wkts_by_bowl.idxmax(), f"({wkts_by_bowl.max()})")
print("6.  Venue with most matches:", venue_counts.idxmax(), f"({venue_counts.max()})")
print("7.  Most common toss decision:", toss_dec.idxmax(), f"({toss_dec.max()/toss_dec.sum()*100:.1f}%)")
print(f"8.  Toss winner won the match {toss_pct}% of the time -> "
      + ("slight advantage only" if toss_pct < 55 else "noticeable advantage"))
print("9.  Season with highest total runs:", season_runs.idxmax(), f"({season_runs.max()})")
print("10. Highest average runs per innings:", team_avg.idxmax(), f"({team_avg.max()})")
print("11. Trend: avg runs per match went from", round(season_avg.iloc[0], 1), "in", season_avg.index[0],
      "to", round(season_avg.iloc[-1], 1), "in", season_avg.index[-1])
print("12. Write 5-10 insights in your own words using the numbers above.")